# Item 04 — Diagnóstico da saturação da ablação (execução no Colab GPU)

Projeto **music-influence-gnn** — marco pós-qualificação, ticket
[`.scratch/next-milestone/issues/04-diagnostico-saturacao-ablacao.md`](https://github.com/cristianomendieta/music-influence-gnn/blob/main/.scratch/next-milestone/issues/04-diagnostico-saturacao-ablacao.md).

**O problema:** a ablação por tipo de aresta devolveu variação de erro **exatamente zero**
para os cinco tipos de aresta *e* para os três grupos de features. Predições bit a bit
idênticas com e sem grafo. Este ticket bloqueia o marco inteiro: nenhum treino novo antes
do veredito.

**As três hipóteses que este notebook separa:**

| | Hipótese | Assinatura esperada |
|---|---|---|
| **A** | O instrumento satura no clamp. `ŷ = clamp(y_prev + Δ, 0, 0,5)`; nas semanas de piso `y_prev = 0` e todo `Δ` negativo é anulado. Com ~95% das semanas no piso, o erro não se move. | Δ varia com o grafo **antes** do clamp; delta_rmse ≈ 0 na leitura completa mas ≠ 0 na leitura on-chart |
| **B** | O grafo não influencia a predição em recorte nenhum. O ganho sobre o SIR viria só do ancoramento à persistência. | Δ é insensível ao grafo mesmo antes do clamp, com embeddings vivos |
| **C** | O instrumento está quebrado: o *harness* da ablação nunca entregou os embeddings ao modelo. | Δ é **constante** entre todas as amostras, e os embeddings da janela chegam zerados ao GRU |

**Já há evidência estática para C**, encontrada na leitura do código em 2026-08-26:
`_predict_all` (em `evaluation/interpretability.py`) chamava
`model.encode_weeks(g, [s.target_week])`, mas `MusicDiffusionGNN.predict` lê o banco nas
semanas da **janela** `[w-W, …, w-1]` — a semana alvo `w` nunca está nela. Nenhuma posição
da janela era encontrada no banco, todas caíam no ramo `torch.zeros(B, hidden)`, o GRU
recebia uma sequência inteiramente nula e `Δ` virava uma constante. Sem dependência do
grafo, a ablação **tem** que dar zero exato — e o mesmo vale para a permutação de features.

Este notebook **não confia nessa leitura**: ele mede as três hipóteses com o checkpoint
real e conclui por um desfecho com o número que o sustenta. As funções de predição são
reimplementadas aqui dentro (autocontido), então o notebook roda mesmo que a correção
do `_predict_all` ainda não esteja no `main`.

**Não há treino aqui.** Só inferência com `results/phase2_experimentos_v2/grid_best_model.pt`
(config `W12_h128_l3_lr5e-04`).

**Saídas** (em `results/item04_diagnostico/`, copiadas ao Drive no fim):
`componentes.parquet`, `variantes.parquet`, `ablacao.parquet`, `embeddings.parquet`,
`DIAGNOSTICO.md`.

## 0. Ambiente — clonar o repositório

Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU (T4)* antes de rodar.
Repo privado: defina `GITHUB_TOKEN` abaixo ou cole um PAT quando pedido.

In [1]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# >>> ajuste conforme seu repositório <<<
REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if not IS_COLAB:
    raise RuntimeError(
        "Este notebook é para rodar no Google Colab (runtime GPU). "
        "Localmente ele roda em CPU, mas as ~8 variantes de grafo ficam lentas."
    )

if not Path(REPO_DIR, "pyproject.toml").exists():
    tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
    if _clone(url).returncode != 0:
        from getpass import getpass
        tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
        _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "torch-geometric", "pyarrow", "tabulate"], check=True)

# o editavel nao entra no sys.path do kernel ja em execucao: aponta direto para src/
sys.path.insert(0, str(Path(REPO_DIR, "src")))

missing = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
print("cwd =", os.getcwd())
if missing:
    raise RuntimeError(f"Faltam dados versionados no repo clonado: {missing}")
print("dados versionados presentes.")

cwd = /content/music-influence-gnn
dados versionados presentes.


## 1. Checkpoint da Phase 2 (`grid_best_model.pt`) — via Google Drive

O checkpoint não é versionado no git. Ele foi salvo no Drive pelo notebook de treino
(`phase2_pipeline_treino.ipynb`) em
`MyDrive/music-influence-gnn/phase2_experimentos_v2/grid_best_model.pt`.

**Só `grid_best_model.pt` serve.** O `best_model.pt` da mesma pasta é outro checkpoint,
mais fraco (`W4_h64_l2`, state_dict cru sem metadados) — a célula abaixo verifica.
Este diagnóstico não usa SIR, então nada de `results/phase0` é necessário.

In [2]:
import shutil
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/music-influence-gnn")
CKPT_SRC = DRIVE_ROOT / "phase2_experimentos_v2" / "grid_best_model.pt"
CKPT_DST = Path(REPO_DIR, "results", "phase2_experimentos_v2", "grid_best_model.pt")
CKPT_DST.parent.mkdir(parents=True, exist_ok=True)

if CKPT_SRC.exists():
    shutil.copy(CKPT_SRC, CKPT_DST)
    print(f"checkpoint copiado de {CKPT_SRC}")
else:
    print(f"NÃO encontrado em {CKPT_SRC} — use o upload manual da célula seguinte.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint copiado de /content/drive/MyDrive/music-influence-gnn/phase2_experimentos_v2/grid_best_model.pt


### 1.1 (Fallback) upload manual do checkpoint

Só rode se a cópia do Drive acima não encontrou o arquivo.

In [3]:
if not CKPT_DST.exists():
    from google.colab import files
    print("Selecione grid_best_model.pt:")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    CKPT_DST.write_bytes(uploaded[fname])
    print(f"salvo em {CKPT_DST}")
else:
    print("checkpoint já presente:", CKPT_DST)

checkpoint já presente: /content/music-influence-gnn/results/phase2_experimentos_v2/grid_best_model.pt


## 2. GPU e configuração do diagnóstico

`TARGET_WEEK_STRIDE` controla o custo: o gasto dominante é um `encode_weeks` do grafo
inteiro por semana **por variante de grafo** (são 8 variantes). Com stride 2 sobre o span
de teste do regime `current` (semanas 208–260) dá ~27 semanas alvo e ~38 encodes por
variante, com o cache de janela aproveitando a sobreposição.

Comece com `SMOKE = True` (2 semanas, 1 variante ablacionada) para validar o encanamento;
depois `SMOKE = False` para a rodada que vale.

In [4]:
import torch, numpy as np, pandas as pd, json, time
from pathlib import Path

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

def md_table(df):
    """DataFrame como tabela markdown; cai para texto puro se tabulate faltar."""
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("AVISO: rodando em CPU — vai ser lento. Ambiente de execução → Alterar tipo de runtime → GPU (T4).")
else:
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} | {_p.total_memory / 1e9:.1f} GB")

# ------------------------------- configuração ------------------------------- #
SMOKE               = False      # True = validação rápida do encanamento
SPLIT_REGIME        = "current" # regime avaliado (o mesmo da qualificação)
TARGET_WEEK_STRIDE  = 2         # 1 = todas as semanas do span de teste
MAX_SONGS           = None      # None = todas as músicas do span de teste
SEED                = 42
EPS                 = 1e-9      # tolerância para "diferença nenhuma"

if SMOKE:
    TARGET_WEEK_STRIDE = 12
    MAX_SONGS = 60

ROOT       = Path(REPO_DIR)
TS_PATH    = ROOT / "data/processed/timeseries.parquet"
GRAPH_PATH = ROOT / "data/processed/graph/hetero_full_current.pt"
NMAP_PATH  = ROOT / "data/processed/graph/node_id_map.json"
CKPT       = ROOT / "results/phase2_experimentos_v2/grid_best_model.pt"
OUT        = ROOT / "results/item04_diagnostico"
OUT.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED)
print(f"SMOKE={SMOKE} | regime={SPLIT_REGIME} | stride={TARGET_WEEK_STRIDE} | max_songs={MAX_SONGS}")

GPU: Tesla T4 | 15.6 GB
SMOKE=False | regime=current | stride=2 | max_songs=None


## 3. Dados, grafo e checkpoint

`pop_bank` é reconstruído a partir da série semanal completa — é ele que fornece
`y_prev`, o valor da persistência ao qual o modelo está ancorado.

In [5]:
from music_diffusion_gnn.training.dataset import (
    aggregate_weekly, build_pop_bank, build_samples, get_split_regime, Sample, _CHART_CODE,
)
from music_diffusion_gnn.evaluation.model_io import load_grid_best_model

t0 = time.time()
ts_df = pd.read_parquet(TS_PATH)
g = torch.load(GRAPH_PATH, weights_only=False)
weekly_df = aggregate_weekly(ts_df)
print(f"timeseries: {ts_df.shape} | weekly: {weekly_df.shape} | music nodes: {g['music'].num_nodes}")

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
assert "state_dict" in ck and "W" in ck, (
    "checkpoint sem metadados — este é o best_model.pt (W4), não o grid_best_model.pt (W12)."
)
W = int(ck["W"])
print(f"checkpoint: {ck['config_str']} | W={W} | val_mse={ck['val_mse']:.6f}")

pop_bank = build_pop_bank(weekly_df, NMAP_PATH, n_music=g["music"].num_nodes)
model = load_grid_best_model(CKPT, g=g, pop_bank_regen=pop_bank, device=DEVICE)
model.eval()
print(f"modelo carregado em {DEVICE} | {time.time()-t0:.1f}s")
print("tipos de aresta:", list(g.edge_types))

timeseries: (4443760, 5) | weekly: (597672, 4) | music nodes: 6526
checkpoint: W12_h128_l3_lr5e-04 | W=12 | val_mse=0.000749
modelo carregado em cuda | 2.6s
tipos de aresta: [('artist', 'performs', 'music'), ('artist', 'has_genre', 'genre'), ('genre', 'rev_has_genre', 'artist'), ('music', 'cotrajectory', 'music'), ('genre', 'cooccurs', 'genre')]


## 4. Amostras de avaliação e o recorte on-chart

As amostras vêm do span de teste do regime, com `first_seen_week` calculado sobre a série
**completa** (se fosse calculado só dentro do teste, a primeira semana de teste de cada
música viraria falsamente a estreia dela e a janela sairia errada).

`onchart` marca `(música, chart, semana)` com `rank_score > 0` em pelo menos um dia — é a
mesma definição usada pelo `run_phase3.py`. O complemento é o **piso**: ausência de
observação, não popularidade baixa.

In [6]:
regime = get_split_regime(SPLIT_REGIME)
MAX_WEEK = 260

def week_index_vec(dates: pd.Series) -> pd.Series:
    iso = dates.dt.isocalendar()
    return (iso["year"].astype(int) - 2017) * 52 + (iso["week"].astype(int) - 1)

# (song_id, chart, week) efetivamente no chart
_oc = ts_df[ts_df["rank_score"] > 0].copy()
_oc["week"] = week_index_vec(_oc["date"]).values
_oc = _oc[(_oc["week"] >= 0) & (_oc["week"] <= MAX_WEEK)]
onchart = set(zip(_oc["song_id"], _oc["chart"], _oc["week"].astype(int)))
print(f"pares (música, chart, semana) on-chart: {len(onchart):,}")

# first_seen GLOBAL, antes de recortar o span de teste
first_seen = (
    weekly_df.groupby(["song_id", "chart"], observed=True)["week"].min().to_dict()
)

test_end = regime.test_end_week if regime.test_end_week is not None else MAX_WEEK
test_wdf = weekly_df[(weekly_df["week"] >= regime.test_start_week) & (weekly_df["week"] <= test_end)].copy()

weeks_all = sorted(test_wdf["week"].unique())
weeks_keep = set(weeks_all[::TARGET_WEEK_STRIDE])
test_wdf = test_wdf[test_wdf["week"].isin(weeks_keep)]

if MAX_SONGS is not None:
    keep_songs = sorted(test_wdf["song_id"].unique())[:MAX_SONGS]
    test_wdf = test_wdf[test_wdf["song_id"].isin(keep_songs)]

samples = build_samples(test_wdf, W=W, node_id_map_path=str(NMAP_PATH), first_seen=first_seen)

# metadados por amostra (a Sample não guarda song_id nem o rótulo do chart)
idx_to_song = {v: k for k, v in json.load(open(NMAP_PATH))["music"]["spotify_id_to_idx"].items()}
code_to_chart = {v: k for k, v in _CHART_CODE.items()}
meta = pd.DataFrame({
    "song_id":     [idx_to_song[s.song_idx] for s in samples],
    "chart":       [code_to_chart[s.chart] for s in samples],
    "target_week": [s.target_week for s in samples],
    "y_true":      [s.y for s in samples],
})
meta["onchart"] = [
    (r.song_id, r.chart, int(r.target_week)) in onchart for r in meta.itertuples()
]

groups: dict[int, list[int]] = {}
for i, s in enumerate(samples):
    groups.setdefault(s.target_week, []).append(i)

print(f"amostras: {len(samples):,} | músicas: {meta.song_id.nunique():,} | semanas alvo: {len(groups)}")
print(f"on-chart: {meta.onchart.mean():.1%}  |  piso: {(~meta.onchart).mean():.1%}")
print(f"y_true=0 (piso pelo alvo): {(meta.y_true == 0).mean():.1%}")

pares (música, chart, semana) on-chart: 44,201
amostras: 98,186 | músicas: 1,955 | semanas alvo: 27
on-chart: 4.6%  |  piso: 95.4%
y_true=0 (piso pelo alvo): 0.0%


## 5. As funções de medição

`predict_components` é a `MusicDiffusionGNN.predict` aberta: devolve `y_prev` (persistência),
`Δ` (a correção que o grafo produz), `y_raw = y_prev + Δ` (antes do clamp) e
`ŷ = clamp(y_raw, 0, 0,5)`. É essa separação que distingue A de B.

Duas diferenças conscientes em relação ao `predict` original:

- a semana de cada posição da janela é tomada como o **máximo** do lote (`max(s.window_weeks[t])`)
  em vez de `samples[0].window_weeks[t]`: dentro de um lote agrupado por semana alvo, as
  posições reais coincidem e só o preenchimento varia por `first_seen_week`, então tomar a
  primeira amostra pode escolher um `-1`;
- `bank_mode="window"` encoda as semanas da janela (o certo, o que `predict` lê) e
  `bank_mode="target"` reproduz o comportamento antigo do harness (encoda só a semana alvo)
  — é a sonda da hipótese C.

In [7]:
from music_diffusion_gnn.graph.temporal import mask_until

@torch.no_grad()
def predict_components(model, bank, samples_batch):
    """Abre MusicDiffusionGNN.predict: retorna y_prev, delta, y_raw, y_hat e
    a fração de posições da janela que o banco de fato forneceu."""
    B = len(samples_batch)
    Wlen = len(samples_batch[0].window_weeks)
    dev = next(model.parameters()).device

    song_idxs = torch.tensor([s.song_idx for s in samples_batch], dtype=torch.long, device=dev)
    seq_parts, pad_cols = [], []
    n_hit = 0
    for t in range(Wlen):
        pad_col = torch.tensor([s.pad_mask[t] for s in samples_batch], dtype=torch.bool, device=dev)
        pad_cols.append(pad_col)
        wk = max(s.window_weeks[t] for s in samples_batch)
        if wk in bank and not pad_col.all():
            emb = bank[wk][song_idxs].masked_fill(pad_col.unsqueeze(-1), 0.0)
            n_hit += 1
        else:
            emb = torch.zeros(B, model.hidden, device=dev)
        seq_parts.append(emb)

    seq = torch.stack(seq_parts, dim=1)
    pad_mask = torch.stack(pad_cols, dim=1)
    delta = model.head(seq, pad_mask)

    prev_weeks = torch.tensor([s.target_week - 1 for s in samples_batch],
                              dtype=torch.long, device=dev).clamp_(min=0)
    chart_codes = torch.tensor([s.chart for s in samples_batch], dtype=torch.long, device=dev)
    y_prev = model.pop_bank[prev_weeks, song_idxs, chart_codes]

    y_raw = y_prev + delta
    return {
        "y_prev": y_prev.cpu().numpy(),
        "delta": delta.cpu().numpy(),
        "y_raw": y_raw.cpu().numpy(),
        "y_hat": y_raw.clamp(0.0, 0.5).cpu().numpy(),
        "window_hit_frac": n_hit / Wlen,
    }


@torch.no_grad()
def run_variant(model, g_var, samples, groups, bank_mode="window", label=""):
    """Roda todas as amostras sobre uma variante do grafo.

    Devolve um DataFrame alinhado 1:1 com `samples` (colunas y_prev, delta, y_raw, y_hat)
    e o dicionário de estatísticas dos embeddings. O cache de embeddings é podado a cada
    semana alvo: só as janelas ainda alcançáveis ficam na memória da GPU.
    """
    zcache: dict[int, torch.Tensor] = {}
    out = {k: np.zeros(len(samples), dtype=np.float64)
           for k in ("y_prev", "delta", "y_raw", "y_hat")}
    hit_fracs, z_stats = [], []

    t0 = time.time()
    for n, week in enumerate(sorted(groups)):
        idxs = groups[week]
        batch = [samples[i] for i in idxs]
        if bank_mode == "window":
            need = sorted({w for s in batch for w in s.window_weeks if w >= 0})
        else:  # "target" — o harness antigo: encoda a semana alvo, que predict nunca lê
            need = [week]

        missing = [w for w in need if w not in zcache]
        if missing:
            zcache.update(model.encode_weeks(g_var, missing))
        bank = {w: zcache[w] for w in need if w in zcache}

        if n == 0 and bank:
            z = next(iter(bank.values()))
            z_stats.append({
                "variant": label,
                "week": int(next(iter(bank))),
                "z_frac_zero": float((z == 0).float().mean()),
                "z_l2_mean": float(z.norm(dim=1).mean()),
                "z_std_between_nodes": float(z.std(dim=0).mean()),
            })

        comp = predict_components(model, bank, batch)
        hit_fracs.append(comp.pop("window_hit_frac"))
        for k, v in comp.items():
            out[k][idxs] = v

        # poda: janelas futuras nunca voltam para semanas < week - W
        for w in [w for w in zcache if w < week - W]:
            del zcache[w]

    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    df = pd.DataFrame(out)
    df["variant"] = label
    print(f"  {label:<28} {time.time()-t0:6.1f}s | posições da janela no banco: {np.mean(hit_fracs):.0%}")
    return df, z_stats


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

## 6. Sonda da hipótese C — o harness entregava embeddings?

Duas rodadas do **mesmo** modelo, sobre o **mesmo** grafo completo, mudando só como o banco
de embeddings é montado. Se `bank_mode="target"` (o jeito antigo) produzir um `Δ` constante,
está provado que a ablação anterior não mediu grafo nenhum — mediu uma constante.

In [8]:
print("rodando as duas montagens de banco sobre o grafo completo:")
df_window, z_window = run_variant(model, g, samples, groups, bank_mode="window", label="full_graph")
df_target, _        = run_variant(model, g, samples, groups, bank_mode="target", label="harness_antigo")

harness = pd.DataFrame([
    {"montagem": "janela (correta)", "delta_std": df_window.delta.std(),
     "delta_min": df_window.delta.min(), "delta_max": df_window.delta.max(),
     "n_valores_distintos": int(pd.unique(np.round(df_window.delta, 12)).size)},
    {"montagem": "semana alvo (antiga)", "delta_std": df_target.delta.std(),
     "delta_min": df_target.delta.min(), "delta_max": df_target.delta.max(),
     "n_valores_distintos": int(pd.unique(np.round(df_target.delta, 12)).size)},
])
display(harness)

HARNESS_QUEBRADO = bool(df_target.delta.std() < EPS and df_window.delta.std() > EPS)
print(f"\nHipótese C (harness quebrado): {'CONFIRMADA' if HARNESS_QUEBRADO else 'não confirmada'}")
if HARNESS_QUEBRADO:
    print(f"  Δ do harness antigo é constante em {df_target.delta.iloc[0]:.6g} "
          f"para todas as {len(df_target):,} amostras — a ablação de jul/2026 não mediu o grafo.")

rodando as duas montagens de banco sobre o grafo completo:
  full_graph                      3.9s | posições da janela no banco: 100%
  harness_antigo                  2.3s | posições da janela no banco: 0%


,montagem,delta_std,delta_min,delta_max,n_valores_distintos
0,janela (correta),0.023389,-0.443126,0.022936,36314
1,semana alvo (antiga),0.000000,0.003204,0.003204,1



Hipótese C (harness quebrado): CONFIRMADA
  Δ do harness antigo é constante em 0.00320397 para todas as 98,186 amostras — a ablação de jul/2026 não mediu o grafo.


## 7. Medições A e B — decomposição `y_prev` vs `Δ`, e o clamp

Quanto da predição é persistência e quanto é a correção estrutural; e com que frequência o
clamp engole a correção. Tudo separado por leitura (on-chart contra piso), porque é
exatamente aí que a hipótese A vive.

In [9]:
comp = pd.concat([meta.reset_index(drop=True), df_window.reset_index(drop=True)], axis=1)
comp["clamp_piso"] = comp.y_raw < 0.0
comp["clamp_teto"] = comp.y_raw > 0.5
comp["clamp_ativo"] = comp.clamp_piso | comp.clamp_teto
comp["delta_engolido"] = comp.y_hat - comp.y_prev          # correção que sobreviveu ao clamp
comp["delta_perdido"] = comp.delta - comp.delta_engolido   # correção anulada pelo clamp

def _resumo(df, nome):
    return {
        "leitura": nome,
        "n": len(df),
        "y_prev_medio": df.y_prev.mean(),
        "abs_delta_medio": df.delta.abs().mean(),
        "abs_delta_mediano": df.delta.abs().median(),
        "delta_sobre_pred": (df.delta.abs().sum() / (df.y_prev.abs().sum() + 1e-12)),
        "frac_delta_nulo": float((df.delta.abs() < 1e-6).mean()),
        "frac_clamp_ativo": float(df.clamp_ativo.mean()),
        "frac_correcao_perdida": float(df.delta_perdido.abs().sum() / (df.delta.abs().sum() + 1e-12)),
    }

decomp = pd.DataFrame([
    _resumo(comp, "todas"),
    _resumo(comp[comp.onchart], "on-chart (principal)"),
    _resumo(comp[~comp.onchart], "piso"),
])
display(decomp.round(6))

for chart in sorted(comp.chart.unique()):
    sub = comp[comp.chart == chart]
    print(f"{chart}: RMSE={rmse(sub.y_true, sub.y_hat):.6f} | "
          f"persistência={rmse(sub.y_true, sub.y_prev):.6f} | "
          f"on-chart {sub.onchart.mean():.1%}")

comp.to_parquet(OUT / "componentes.parquet")

,leitura,n,y_prev_medio,abs_delta_medio,abs_delta_mediano,delta_sobre_pred,frac_delta_nulo,frac_clamp_ativo,frac_correcao_perdida
0,todas,98186,0.014892,0.010728,0.008375,0.720401,0.000020,0.827633,0.768980
1,on-chart (principal),4543,0.282608,0.023222,0.010902,0.082169,0.000000,0.052829,0.036421
2,piso,93643,0.001904,0.010122,0.008225,5.317153,0.000021,0.865222,0.850515


top200: RMSE=0.023369 | persistência=0.022856 | on-chart 7.5%
viral50: RMSE=0.025083 | persistência=0.026385 | on-chart 1.7%


## 8. Medição C — o grafo muda a predição **antes** do clamp?

Oito variantes do mesmo modelo, mudando só a estrutura:

- `full_graph` — referência;
- uma ablação por tipo de aresta (o tipo é esvaziado, zero arestas, sobre uma cópia do grafo);
- `rewired` — cotrajetória religada ao acaso preservando o número de arestas (prévia barata
  do item 08; se a predição não mexe nem aqui, a topologia não está sendo usada).

A comparação decisiva é sobre `Δ` e `y_raw`, **antes** do clamp. Se aí não muda nada, o
clamp é inocente e o desfecho é B.

In [10]:
from music_diffusion_gnn.evaluation.interpretability import _empty_edge_store

def rewire(g_in, edge_type, seed=SEED):
    """Religa um tipo de aresta ao acaso, preservando a contagem de arestas e os nós de origem."""
    g_out = g_in.clone()
    store = g_out[edge_type]
    ei = store.edge_index
    n_dst = g_out[edge_type[2]].num_nodes
    gen = torch.Generator(device="cpu").manual_seed(seed)
    store.edge_index = torch.stack(
        [ei[0], torch.randint(0, n_dst, (ei.shape[1],), generator=gen).to(ei.device)]
    )
    return g_out

COTRAJ = ("music", "cotrajectory", "music")
variantes = {}
for et in g.edge_types:
    if SMOKE and et != COTRAJ:
        continue
    variantes[f"sem_{et[1]}[{et[0]}->{et[2]}]"] = _empty_edge_store(g, et)
variantes["rewired_cotrajectory"] = rewire(g, COTRAJ)

print(f"{len(variantes)} variantes além da referência:")
frames, zstats = [df_window], list(z_window)
for label, g_var in variantes.items():
    try:
        d, zs = run_variant(model, g_var, samples, groups, bank_mode="window", label=label)
        frames.append(d); zstats.extend(zs)
    except KeyError as e:
        # HeteroConv só devolve os tipos de nó que receberam mensagem; se 'music'
        # ficar sem nenhuma aresta de entrada, o encoder não produz saída.
        print(f"  {label:<28} SEM SAÍDA do encoder ({e}) — 'music' ficou sem aresta de entrada")

var_df = pd.concat(frames, ignore_index=True)
var_df.to_parquet(OUT / "variantes.parquet")
pd.DataFrame(zstats).to_parquet(OUT / "embeddings.parquet")
display(pd.DataFrame(zstats).round(6))

6 variantes além da referência:
  sem_performs[artist->music]     3.0s | posições da janela no banco: 100%
  sem_has_genre[artist->genre]    2.9s | posições da janela no banco: 100%
  sem_rev_has_genre[genre->artist]    3.4s | posições da janela no banco: 100%
  sem_cotrajectory[music->music]    1.9s | posições da janela no banco: 100%
  sem_cooccurs[genre->genre]      2.8s | posições da janela no banco: 100%
  rewired_cotrajectory            2.9s | posições da janela no banco: 100%


,variant,week,z_frac_zero,z_l2_mean,z_std_between_nodes
0,full_graph,196,0.901546,0.752687,0.010742
1,sem_performs[artist->music],196,0.880777,0.763284,0.010173
2,sem_has_genre[artist->genre],196,0.899164,0.759687,0.010628
3,sem_rev_has_genre[genre->artist],196,0.896204,0.765384,0.010561
4,sem_cotrajectory[music->music],196,0.865881,0.727279,0.007681
5,sem_cooccurs[genre->genre],196,0.901546,0.752687,0.010742
6,rewired_cotrajectory,196,0.934796,0.806975,0.002233


In [11]:
base = df_window.reset_index(drop=True)
linhas = []
for label in [l for l in var_df.variant.unique() if l != "full_graph"]:
    v = var_df[var_df.variant == label].reset_index(drop=True)
    d_delta = (v.delta - base.delta).abs()
    d_raw   = (v.y_raw - base.y_raw).abs()
    d_hat   = (v.y_hat - base.y_hat).abs()
    oc = meta.onchart.values
    linhas.append({
        "variante": label,
        "frac_delta_muda": float((d_delta > EPS).mean()),
        "delta_dif_media": float(d_delta.mean()),
        "delta_dif_max": float(d_delta.max()),
        "frac_pred_muda_pos_clamp": float((d_hat > EPS).mean()),
        "pred_dif_media_pos_clamp": float(d_hat.mean()),
        "frac_pred_muda_onchart": float((d_hat[oc] > EPS).mean()) if oc.any() else float("nan"),
        "frac_pred_muda_piso": float((d_hat[~oc] > EPS).mean()) if (~oc).any() else float("nan"),
    })
sensib = pd.DataFrame(linhas)
display(sensib.round(8))

GRAFO_IMPORTA_PRE_CLAMP = bool((sensib.frac_delta_muda > 0.01).any())
print(f"\nO grafo altera Δ antes do clamp: {'SIM' if GRAFO_IMPORTA_PRE_CLAMP else 'NÃO'}")

,variante,frac_delta_muda,delta_dif_media,delta_dif_max,frac_pred_muda_pos_clamp,pred_dif_media_pos_clamp,frac_pred_muda_onchart,frac_pred_muda_piso
0,sem_performs[artist->music],0.991160,0.005872,1.910736e-01,0.339437,0.000754,0.944310,0.310093
1,sem_has_genre[artist->genre],0.972888,0.001556,1.080600e-01,0.167050,0.000162,0.911072,0.130955
2,sem_rev_has_genre[genre->artist],0.972868,0.002782,1.328642e-01,0.181523,0.000335,0.917015,0.145841
3,sem_cotrajectory[music->music],0.869207,0.019303,3.719485e-01,0.824150,0.008611,0.933304,0.818855
4,sem_cooccurs[genre->genre],0.306113,0.000000,3.000000e-08,0.007211,0.000000,0.081224,0.003620
5,rewired_cotrajectory,1.000000,0.004734,4.249189e-01,0.174414,0.000976,0.952895,0.136647



O grafo altera Δ antes do clamp: SIM


## 9. Medição D — a ablação refeita, nas três leituras

`delta_rmse` contra a referência, em três leituras: completa, on-chart (a principal) e
pré-clamp (`y_raw`, que ignora a saturação por construção). Um `delta_rmse` que só aparece
nas duas últimas é a assinatura da hipótese A.

In [12]:
y_true = meta.y_true.values
oc = meta.onchart.values

def _rmse_set(v):
    return {
        "rmse_full": rmse(y_true, v.y_hat),
        "rmse_onchart": rmse(y_true[oc], v.y_hat.values[oc]) if oc.any() else float("nan"),
        "rmse_pre_clamp": rmse(y_true, v.y_raw),
        "rmse_pre_clamp_onchart": rmse(y_true[oc], v.y_raw.values[oc]) if oc.any() else float("nan"),
    }

ref = _rmse_set(base)
rows = []
for label in var_df.variant.unique():
    v = var_df[var_df.variant == label].reset_index(drop=True)
    r = _rmse_set(v)
    rows.append({
        "componente": label,
        **{f"delta_{k}": r[k] - ref[k] for k in ref},
        **{k: r[k] for k in ref},
    })
abl = pd.DataFrame(rows)
abl.to_parquet(OUT / "ablacao.parquet")
display(abl.round(8))

nz = abl[abl.componente != "full_graph"]
ABLACAO_VIVA_ONCHART = bool((nz.delta_rmse_onchart.abs() > EPS).any())
ABLACAO_VIVA_FULL    = bool((nz.delta_rmse_full.abs() > EPS).any())
print(f"ablação sensível na leitura completa: {ABLACAO_VIVA_FULL}")
print(f"ablação sensível no recorte on-chart: {ABLACAO_VIVA_ONCHART}")

,componente,delta_rmse_full,delta_rmse_onchart,delta_rmse_pre_clamp,delta_rmse_pre_clamp_onchart,rmse_full,rmse_onchart,rmse_pre_clamp,rmse_pre_clamp_onchart
0,full_graph,0.000000,0.000000,0.000000,0.000000,0.024241,0.100487,0.034149,0.102311
1,sem_performs[artist->music],-0.000095,-0.002506,-0.005544,-0.003282,0.024146,0.097981,0.028606,0.099029
2,sem_has_genre[artist->genre],-0.000054,-0.000763,-0.002497,-0.000969,0.024187,0.099724,0.031652,0.101342
3,sem_rev_has_genre[genre->artist],-0.000104,-0.001623,-0.003598,-0.002080,0.024137,0.098864,0.030551,0.100230
4,sem_cotrajectory[music->music],0.018048,0.093292,0.025133,0.103178,0.042289,0.193779,0.059283,0.205489
5,sem_cooccurs[genre->genre],0.000000,0.000000,0.000000,0.000000,0.024241,0.100487,0.034149,0.102311
6,rewired_cotrajectory,-0.000150,-0.002840,-0.008468,-0.003867,0.024091,0.097647,0.025682,0.098444


ablação sensível na leitura completa: True
ablação sensível no recorte on-chart: True


## 10. Veredito

A regra é declarada aqui, e não depois de ver os números:

- **C** — se o `Δ` do harness antigo é constante enquanto o corrigido varia: a ablação de
  jul/2026 não mediu o grafo, e o zero exato está explicado pelo instrumento. O veredito
  A/B passa então a valer sobre a ablação **refeita** neste notebook.
- **A** — o grafo altera `Δ` antes do clamp e a ablação refeita move o erro no recorte
  on-chart: o instrumento saturava, o recorte devolve sensibilidade, o plano do marco segue.
- **B** — o grafo não altera `Δ` nem antes do clamp: a estrutura não influencia a predição,
  e o ganho da GNN sobre o SIR vem do ancoramento à persistência — o que muda a leitura de
  toda a avaliação publicada na qualificação.

In [13]:
if not GRAFO_IMPORTA_PRE_CLAMP:
    desfecho = "B"
    frase = ("O grafo não altera a predição nem antes do clamp. A estrutura relacional não "
             "influencia a saída do modelo; o ganho sobre o SIR vem do ancoramento à "
             "persistência. A leitura da avaliação da qualificação precisa ser revista.")
elif ABLACAO_VIVA_ONCHART or ABLACAO_VIVA_FULL:
    desfecho = "A"
    frase = ("O grafo altera a predição e a ablação refeita move o erro. O instrumento "
             "anterior saturava; o recorte on-chart devolve sensibilidade. O plano do marco segue.")
else:
    desfecho = "A-parcial"
    frase = ("O grafo altera Δ antes do clamp, mas a correção não sobrevive ao clamp em "
             "nenhuma leitura: a saturação é total no conjunto avaliado. Segue o plano, "
             "com a ressalva de que a ablação só terá sensibilidade se o clamp for revisto.")

linhas_md = [
    "# Item 04 — Diagnóstico da saturação da ablação",
    "",
    f"- checkpoint: `{ck['config_str']}` (W={W}, val_mse={ck['val_mse']:.6f})",
    f"- regime de split: `{SPLIT_REGIME}` | semanas alvo: {len(groups)} (stride {TARGET_WEEK_STRIDE})",
    f"- amostras: {len(samples):,} | músicas: {meta.song_id.nunique():,} | "
    f"on-chart: {meta.onchart.mean():.1%} | piso: {(~meta.onchart).mean():.1%}",
    f"- SMOKE: {SMOKE}",
    "",
    f"## Veredito: desfecho {desfecho}",
    "",
    frase,
    "",
    f"Hipótese C (harness da ablação quebrado): **{'confirmada' if HARNESS_QUEBRADO else 'não confirmada'}**.",
    "",
    "## Harness: montagem do banco de embeddings", "", md_table(harness.round(8)), "",
    "## Decomposição y_prev vs Δ e clamp", "", md_table(decomp.round(6)), "",
    "## Sensibilidade ao grafo (antes e depois do clamp)", "", md_table(sensib.round(8)), "",
    "## Ablação refeita — delta_rmse por leitura", "", md_table(abl.round(8)), "",
    "## Embeddings por variante", "", md_table(pd.DataFrame(zstats).round(6)), "",
]
(OUT / "DIAGNOSTICO.md").write_text("\n".join(linhas_md), encoding="utf-8")

print("=" * 70)
print(f"DESFECHO {desfecho}")
print(frase)
print("=" * 70)
print((OUT / "DIAGNOSTICO.md").read_text()[:2000])

DESFECHO A
O grafo altera a predição e a ablação refeita move o erro. O instrumento anterior saturava; o recorte on-chart devolve sensibilidade. O plano do marco segue.
# Item 04 — Diagnóstico da saturação da ablação

- checkpoint: `W12_h128_l3_lr5e-04` (W=12, val_mse=0.000749)
- regime de split: `current` | semanas alvo: 27 (stride 2)
- amostras: 98,186 | músicas: 1,955 | on-chart: 4.6% | piso: 95.4%
- SMOKE: False

## Veredito: desfecho A

O grafo altera a predição e a ablação refeita move o erro. O instrumento anterior saturava; o recorte on-chart devolve sensibilidade. O plano do marco segue.

Hipótese C (harness da ablação quebrado): **confirmada**.

## Harness: montagem do banco de embeddings

| montagem             |   delta_std |   delta_min |   delta_max |   n_valores_distintos |
|:---------------------|------------:|------------:|------------:|----------------------:|
| janela (correta)     |   0.0233889 | -0.443126   |  0.0229356  |                 36314 |
| semana alvo (ant

## 11. Persistir no Drive

Copia `results/item04_diagnostico/` para o Drive. A trava aborta se a rodada foi um smoke —
smoke valida encanamento, não sustenta veredito.

In [14]:
assert not SMOKE, (
    "ABORTADO: esta foi uma rodada SMOKE. Volte à célula de configuração, ponha "
    "SMOKE = False, rode o notebook inteiro de novo e só então persista."
)

DRIVE_OUT = DRIVE_ROOT / "item04_diagnostico"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for f in OUT.iterdir():
    shutil.copy(f, DRIVE_OUT / f.name)
print(f"copiado para {DRIVE_OUT}:")
for f in sorted(DRIVE_OUT.iterdir()):
    print(f"  {f.name}  {f.stat().st_size/1024:.1f} KB")

copiado para /content/drive/MyDrive/music-influence-gnn/item04_diagnostico:
  DIAGNOSTICO.md  6.3 KB
  ablacao.parquet  7.0 KB
  componentes.parquet  1145.2 KB
  embeddings.parquet  4.1 KB
  variantes.parquet  4828.8 KB


## 12. Depois de rodar (localmente)

1. Baixar `item04_diagnostico/` do Drive para `results/item04_diagnostico/` no repo local.
2. Registrar o veredito no ticket 04 e em `.scratch/next-milestone/CHECKLIST.md`.
3. Se **A** ou **A-parcial**: destravar o item 05 (atributos estruturais de gênero) e seguir
   a escada de comparação. Se **B**: parar a escada e reescrever a leitura da avaliação,
   conforme o pré-compromisso da [ADR-0001](docs/adr/0001-precompromisso-de-falseamento.md).
4. Se a hipótese **C** for confirmada, a correção do `_predict_all` já está no
   `evaluation/interpretability.py` — o `run_phase3.py` volta a produzir uma ablação que
   mede alguma coisa, e a ablação de jul/2026 vira nota de rodapé, não resultado.